In [1]:
import os, gc, importlib.util
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
VOL = {"v1": "/tmp/vol_test", "v2": "/tmp/vol_test_v2"}

# indeks semua file input sekali saja (lewati folder kompetisi & file volume .npz)
FILES = {}
for root, dirs, files in os.walk("/kaggle/input"):
    dirs[:] = [d for d in dirs if d != "competitions"]
    for f in files:
        if not f.endswith(".npz"):
            FILES.setdefault(f, os.path.join(root, f))

def find_file(name):
    if name not in FILES:
        raise FileNotFoundError(name)
    return FILES[name]

def load_module(name, fname):
    path = find_file(fname)
    print(f"{fname:28s} -> {path}")
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

pp1 = load_module("pp1", "03_preprocess_mri.py")       # preprocessing v1 (3 view, 224)
pp2 = load_module("pp2", "03b_preprocess_mri_v2.py")   # preprocessing v2 (5 view, 256)
tb  = load_module("tb", "09_train_student.py")         # KneeNet versi baru: ViT otomatis resize 256 -> 224
LABELS = tb.LABELS

# nama: (backbone, file bobot, versi data). "kf5" = rata-rata 5 model fold sebagai satu anggota
MODELS = {
    "effb0":             ("efficientnet_b0", ["efficientnet_b0_best.pt"], "v1"),
    "effb0_s2":          ("efficientnet_b0", ["effb0_s2_best.pt"], "v1"),
    "dinov2s_v2":        ("vit_small_patch14_dinov2.lvd142m", ["dinov2s_v2_best.pt"], "v1"),
    "student_v2":        ("efficientnet_b0", ["effb0_student_v2_best.pt"], "v2"),
    "student_v2_s7":     ("efficientnet_b0", ["effb0_student_v2_s7_best.pt"], "v2"),
    "dinov2_student_v2": ("vit_small_patch14_dinov2.lvd142m", ["dinov2s_student_v2_best.pt"], "v2"),
    "kf5":               ("efficientnet_b0", [f"effb0_kf_f{i}_best.pt" for i in range(5)], "v1"),
}

USE = ["effb0", "dinov2s_v2", "student_v2", "student_v2_s7", "dinov2_student_v2"]   # opsi 3: val 0.8463 | gold 0.8572
# USE = ["dinov2s_v2", "student_v2", "student_v2_s7", "dinov2_student_v2"]          # opsi 1: val 0.8470 | gold 0.8517

for name in USE:
    for w in MODELS[name][1]:
        print(f"{name:17s} {w:30s} -> {find_file(w)}")
print("Data yang perlu diproses:", sorted({MODELS[n][2] for n in USE}))

03_preprocess_mri.py         -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/03_preprocess_mri.py
03b_preprocess_mri_v2.py     -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-02b-preprocess-mri-v2/03b_preprocess_mri_v2.py
09_train_student.py          -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-08-student-dinov2-v2data/09_train_student.py
effb0             efficientnet_b0_best.pt        -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03-train-baseline/efficientnet_b0_best.pt
dinov2s_v2        dinov2s_v2_best.pt             -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-03c-train-dinov2-v2/dinov2s_v2_best.pt
student_v2        effb0_student_v2_best.pt       -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-07-tahapc-student/effb0_student_v2_best.pt
student_v2_s7     effb0_student_v2_s7_best.pt    -> /kaggle/input/notebooks/kellenvalerie/rsna-knee-08-student-dinov2-v2data/effb0_student_v2_s7_best.pt
dinov2_student_v2 dinov2s_student_v2_best.pt     -> 

In [2]:
series = pd.read_csv(f"{COMP}/test_series.csv")
test = pd.read_csv(f"{COMP}/test.csv")
uids = test["StudyInstanceUID"].tolist()
need = {MODELS[n][2] for n in USE}

avail = {}
for ver, pp in [("v1", pp1), ("v2", pp2)]:
    if ver not in need:
        continue
    pp.SPLIT, pp.OUT_DIR = "test", VOL[ver]
    os.makedirs(VOL[ver], exist_ok=True)
    failed = 0
    for uid, g in tqdm(series.groupby("StudyInstanceUID"), desc=f"Preprocess {ver}"):
        try:
            pp.process_study(uid, g)
        except Exception as e:
            failed += 1
            print("Gagal:", ver, uid, type(e).__name__, e)
    avail[ver] = [u for u in uids if os.path.exists(f"{VOL[ver]}/{u}.npz")]
    print(f"{ver}: volume tersedia {len(avail[ver])}/{len(uids)} | gagal {failed}")
    assert len(avail[ver]) >= 0.95 * len(uids), f"Terlalu banyak studi gagal ({ver}), jangan disubmit"

Preprocess v1:   0%|          | 0/3 [00:00<?, ?it/s]

v1: volume tersedia 3/3 | gagal 0


Preprocess v2:   0%|          | 0/3 [00:00<?, ?it/s]

v2: volume tersedia 3/3 | gagal 0


In [3]:
member_preds = {}
for name in USE:
    backbone, wfiles, ver = MODELS[name]
    ok = avail[ver]
    n_views = int(np.load(f"{VOL[ver]}/{ok[0]}.npz")["mask"].shape[0])
    ds = tb.KneeDS(ok, np.full((len(ok), len(LABELS)), np.nan, np.float32), VOL[ver], train=False)
    dl = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=False, num_workers=0)

    ps = []
    for w in wfiles:
        net = tb.KneeNet(backbone, n_views=n_views, pretrained=False)   # struktur saja, tanpa download
        net.load_state_dict(torch.load(find_file(w), map_location="cpu"))
        net = net.cuda().eval()
        if "vit" not in backbone:
            net = net.to(memory_format=torch.channels_last)
        ps.append(tb.predict(net, dl))
        del net; gc.collect(); torch.cuda.empty_cache()

    p = pd.DataFrame(np.mean(ps, axis=0), columns=LABELS)
    p.insert(0, "StudyInstanceUID", ok)
    member_preds[name] = test[["StudyInstanceUID"]].merge(p, on="StudyInstanceUID", how="left")[LABELS].values
    print(f"{name:11s} ({len(wfiles)} bobot, data {ver}, {n_views} view) selesai | "
          f"rata-rata prediksi {np.nanmean(member_preds[name]):.3f}")

p_ens = np.nanmean(np.stack([member_preds[n] for n in USE]), axis=0)
for n in USE:
    pd.DataFrame(member_preds[n], columns=LABELS).assign(StudyInstanceUID=uids).to_csv(f"/kaggle/working/test_preds_{n}.csv", index=False)
print("Anggota ensemble:", USE, "| bentuk:", p_ens.shape)

effb0       (1 bobot, data v1, 3 view) selesai | rata-rata prediksi 0.182
dinov2s_v2  (1 bobot, data v1, 3 view) selesai | rata-rata prediksi 0.232
student_v2  (1 bobot, data v2, 5 view) selesai | rata-rata prediksi 0.174
student_v2_s7 (1 bobot, data v2, 5 view) selesai | rata-rata prediksi 0.178
dinov2_student_v2 (1 bobot, data v2, 5 view) selesai | rata-rata prediksi 0.189
Anggota ensemble: ['effb0', 'dinov2s_v2', 'student_v2', 'student_v2_s7', 'dinov2_student_v2'] | bentuk: (3, 12)


In [4]:
sub = pd.DataFrame(p_ens, columns=LABELS)
sub.insert(0, "StudyInstanceUID", uids)
n_missing = sub[LABELS].isna().any(axis=1).sum()
sub[LABELS] = sub[LABELS].fillna(0.5)

sample = pd.read_csv(f"{COMP}/sample_submission.csv")
sub = sub[sample.columns]
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Studi tanpa prediksi (diisi 0.5):", n_missing)
print("Shape submission:", sub.shape, "| shape sample:", sample.shape)
print(sub.head())

Studi tanpa prediksi (diisi 0.5): 0
Shape submission: (3, 13) | shape sample: (3, 13)
                                    StudyInstanceUID       ACL       MCL  \
0  1.2.826.0.1.3680043.8.498.10047035057544427318...  0.002600  0.042306   
1  1.2.826.0.1.3680043.8.498.10062861783145312629...  0.384807  0.061545   
2  1.2.826.0.1.3680043.8.498.10067514707072572280...  0.004264  0.015877   

   Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  \
0         0.250008          0.069128   0.227332    0.126840  0.302197   
1         0.418748          0.358150   0.090316    0.124571  0.199827   
2         0.455675          0.109662   0.320390    0.132805  0.662387   

   Effusion  Synovitis   Baker's  Contusion  Fracture  
0  0.532336   0.042157  0.665249   0.018919  0.003153  
1  0.171591   0.458155  0.093961   0.150137  0.077770  
2  0.106470   0.022373  0.150851   0.012197  0.015046  
